## Allocation test version 1
This test doesn't import the module with the functions to allocate crews and create controls, instead has the block
of code of those functions written within this notebook.

In [1]:
#import sys
#!{sys.executable} -m pip install import_ipynb

In [2]:
#import import_ipynb
#from BPDRR_crew_allocation import allocate_crews
from pprint import pprint
import random
import pandas as pd
import numpy as np

In [3]:
def allocate_crews(reparations, dmatrix, indexes, n_teams):
    missing_repairs = set(indexes) - set(reparations)
    if missing_repairs:
        raise KeyError(f"IDs missing from reparations: {sorted(missing_repairs)}")

    missing_matrix = set(indexes) - set(dmatrix.index)
    if missing_matrix:
        raise KeyError(f"IDs missing from travel-time matrix: {sorted(missing_matrix)}")

    # existing allocation logic...
    """
    allocate_crews(reparations, dmatrix, indexes, n_teams):
    
    Description
       This is essentially a greedy load-balancing problem: 
       process jobs in the order given by indexes, and always assign the next job 
       to the crew with the smallest accumulated repair time.
    
    Input Parameters
    -----------------
    reparations : dict
        Dictionary {id: repair_time}
    dmatrix : dict
        Dictionary {id_i: {id_j: travel_time}}
    indexes : list
        List of ids indicating the allocation order, must have the same length of the keys of 'reparations'
    n_teams : int
        Number of crews, how many crews are you sending in the field to repair pipes

    Output / Returns
    ----------------
    dictionary with the pipe ids and the times of interventions for each crew. 
    dict { 'crew_1': { 'pipe_ids': [...], 'time_total': ...},
           'crew_2': { 'pipe_ids': [...], 'time_total': ...}, 
           ...
         }

    Developed by : Mario Castro-Gama, ir. MSc. PhD
    Last update  : 2026-05-20
                   2026-06-15, added travel time constant
                   2026-07-21, added travel tiem as function of 'dmatrix' distance matrix
    
    """

    # if each crew key is a string
    # crews = { f"crew_{i+1}": {'pipe_ids':  [], 
    #                           'time_k':    [], 
    #                           'time_t':    [], 
    #                           'time_0':    [],
    #                           'time_1':    [],
    #                           'time_total': 0,
    #                          } for i in range(n_teams)}

    # if each crew key is an int (starting at 1)
    crews = { i+1: {'pipe_ids':  [], 
                    'time_repair':    [], 
                    'time_travel':   [],
                    'time_0':    [],
                    'time_1':    [],
                    'time_total': 0,
                   } for i in range(n_teams)}

    nrep = len(reparations)

    print('')
    print('organize distances for each crew')
    new_controls = []
    for idx in indexes:
        if idx not in reparations:
            raise KeyError(f"Pipe ID '{idx}' not found in reparations")

        # Find the crew with the minimum accumulated time to allocate the next reparation
        # first available gets chosen
        crew_curr = min(crews, key = lambda c: crews[c]['time_total'])

        # current reparation time
        repair_time = reparations[idx]['t_r']

        # find the travel time between this pipe and the previous one
        if crews[crew_curr]['pipe_ids']==[]:
            travel_time = 0.5  # no previous pipe so give it 30 minutes
        else:
            # This is estimated from distance matrix 'dmatrix', that matrix is square and static for each damage scenario
            pipe_prev = crews[crew_curr]['pipe_ids'][-1]
            travel_time = dmatrix[pipe_prev][idx]
            print('Crew '+str(crew_curr)+', from '+pipe_prev+'-to-'+idx+' : '+str(travel_time))
        
        # Assign the reparation to each crew
        crews[crew_curr]['pipe_ids'].append(idx)
        crews[crew_curr]['time_repair'].append(repair_time)
        crews[crew_curr]['time_travel'].append(travel_time)
        crews[crew_curr]['time_0'].append(travel_time + crews[crew_curr]['time_total'])
        crews[crew_curr]['time_1'].append(repair_time + crews[crew_curr]['time_0'][-1])
        crews[crew_curr]['time_total'] += repair_time + travel_time

        new_controls.append(f"; Crew {crew_curr} - Pipe {idx}\n")
        new_controls.append(f"LINK {idx}_A CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<A> when arriving to the location
        new_controls.append(f"LINK {idx}_B CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<B> when arriving to the location
        new_controls.append(f"LINK {idx} OPEN AT TIME {crews[crew_curr]['time_1'][-1]}\n")     # Open Pipe_id only after time of reparation
        
    return crews, new_controls

In [4]:
# Damage scenario selection
ds_sel = 'DS5'

df = pd.read_excel(
    "DS_with_full_description.xlsx",
    sheet_name= ds_sel
)

# Repair time discretization (hours)
repair_time_interval = 0.25  # 15 minutes

# Build the reparations dictionary
time_reparation = {}

for _, row in df.iterrows():

    repair_time = float(row["fix time (hours)"])

    # Round up to the nearest interval
    repair_time = np.ceil(repair_time / repair_time_interval) * repair_time_interval

    time_reparation[str(row["Pipe ID"])] = {
        "t_r": repair_time
    }

print(f"{len(time_reparation)} repairs loaded.")
#print(list(time_reparation.items())[5:10])
time_reparation

108 repairs loaded.


{'3922': {'t_r': np.float64(7.25)},
 '3398': {'t_r': np.float64(7.25)},
 '3825': {'t_r': np.float64(7.25)},
 '4117': {'t_r': np.float64(7.25)},
 '1902': {'t_r': np.float64(5.75)},
 '3019': {'t_r': np.float64(5.75)},
 '91': {'t_r': np.float64(5.75)},
 '1540': {'t_r': np.float64(4.5)},
 '2050': {'t_r': np.float64(4.5)},
 '2115': {'t_r': np.float64(4.5)},
 '378': {'t_r': np.float64(4.5)},
 '4129': {'t_r': np.float64(4.5)},
 '1028': {'t_r': np.float64(3.5)},
 '4594': {'t_r': np.float64(3.5)},
 '2162': {'t_r': np.float64(3.5)},
 '5105': {'t_r': np.float64(3.5)},
 '1402': {'t_r': np.float64(3.5)},
 '4652': {'t_r': np.float64(3.5)},
 '1282': {'t_r': np.float64(3.5)},
 '1631': {'t_r': np.float64(3.5)},
 '2404': {'t_r': np.float64(3.5)},
 '246': {'t_r': np.float64(3.5)},
 '2804': {'t_r': np.float64(3.5)},
 '3136': {'t_r': np.float64(3.5)},
 '3807': {'t_r': np.float64(3.5)},
 '4748': {'t_r': np.float64(3.5)},
 '475': {'t_r': np.float64(3.5)},
 '5004': {'t_r': np.float64(3.5)},
 '5458': {'t_r': n

## Example 1
This example uses a dmatrix that uses ramdom values

In [5]:
#n_teams = 3

# Pipe IDs from the reparations dictionary
#pipe_ids = list(time_reparation.keys())

# Number of repairs
#n_rep = len(pipe_ids)

# Random travel times between 0 and 2 hours
#random_matrix = np.random.uniform(0, 2, (n_rep, n_rep))

# Travel from a pipe to itself is zero
#np.fill_diagonal(random_matrix, 0)

# Create DataFrame with pipe IDs
#dmatrix_df = pd.DataFrame(
#    random_matrix,
#    index=pipe_ids,
#    columns=pipe_ids
#)

#dmatrix_df.head()

In [6]:
# generate a random permutation, one would expect to get this directly from the optimization (PYMOO)
#indexes = random.sample(pipe_ids, n_rep)
#print('Show permutation of reparations')
#print(indexes)
#print(len(indexes))

In [7]:
# apply the greedy allocation to the dataset
#crews, new_controls = allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
#print('')
#print('[CONTROLS]')
#pprint(new_controls)

In [8]:
#pd.DataFrame({"crews": new_controls}).to_excel(
#    "new_controls.xlsx",
#    index=False
#)

## Example 2
This example uses a dmatrix imported from the files generated by "BPDRR_travel_time_matrix_gen.ipynb"

In [9]:
# Number of crews
n_teams = 6

# Pipe IDs from the reparations dictionary
pipe_ids = list(time_reparation.keys())

# Number of repairs
n_rep = len(pipe_ids)

In [10]:
# Import the travel time matrix from the Excel file
dmatrix_df = pd.read_parquet(
    "TravelTime_"+ds_sel+".parquet"
)

# Normalize IDs so the repair dictionary and matrix use the same keys
dmatrix_df.index = dmatrix_df.index.map(str)
dmatrix_df.columns = dmatrix_df.columns.map(str)

dmatrix_df.head()

,3922,3398,3825,4117,1902,3019,91,1540,2050,2115,...,5280,5432,5523,5550,5845,587,5893,772,90,98
3922,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.75,0.75,0.25,0.75,0.25,0.25,0.25
3398,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.50,0.50,0.50,0.25,0.50,0.25,0.25,0.25
3825,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.50,0.50,0.25,0.50,0.25,0.25,0.25
4117,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.75,0.75,0.25,0.75,0.25,0.50,0.25
1902,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.50,0.25,0.25,...,0.75,0.75,0.75,0.50,0.75,0.25,0.75,0.25,0.25,0.25


In [11]:
indexes = random.sample(pipe_ids, n_rep)
print('Show permutation of reparations')
print(indexes)
print(len(indexes))

Show permutation of reparations
['3723', '2600', '1631', '154', '4805', '5458', '6005', '3807', '3762', '253', '4117', '5550', '721', '3136', '5100', '153', '2596', '5280', '378', '1926', '6012', '3842', '1731', '4889', '5523', '3398', '838', '1857', '88', '246', '5865', '3613', '3754', '3019', '1402', '4379', '90', '5022', '3825', '4129', '2115', '3409', '2646', '3690', '3216', '2464', '5688', '772', '2162', '626', '1832', '3049', '5432', '2804', '3123', '2050', '98', '1936', '809', '1558', '3615', '1989', '475', '5105', '5004', '4386', '2626', '3547', '587', '335', '1794', '4594', '1947', '3720', '2651', '1282', '4652', '2068', '4065', '4316', '1352', '3536', '4671', '5845', '1007', '1028', '91', '5207', '1418', '3121', '4093', '5194', '2772', '1902', '1640', '2404', '60', '2972', '390', '3922', '5893', '4022', '2212', '1060', '3800', '1540', '4748', '1032']
108


In [12]:
# apply the greedy allocation to the dataset
crews, new_controls = allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
print('')
print('[CONTROLS]')
pprint(new_controls)


organize distances for each crew
Crew 2, from 2600-to-6005 : 0.75
Crew 4, from 154-to-3807 : 0.25
Crew 3, from 1631-to-3762 : 0.5
Crew 6, from 5458-to-253 : 0.5
Crew 1, from 3723-to-4117 : 0.25
Crew 4, from 3807-to-5550 : 0.75
Crew 3, from 3762-to-721 : 0.25
Crew 5, from 4805-to-3136 : 0.5
Crew 6, from 253-to-5100 : 0.5
Crew 2, from 6005-to-153 : 0.5
Crew 4, from 5550-to-2596 : 0.75
Crew 6, from 5100-to-5280 : 0.25
Crew 3, from 721-to-378 : 0.25
Crew 5, from 3136-to-1926 : 0.25
Crew 2, from 153-to-6012 : 0.5
Crew 1, from 4117-to-3842 : 0.25
Crew 6, from 5280-to-1731 : 0.75
Crew 4, from 2596-to-4889 : 0.75
Crew 1, from 3842-to-5523 : 0.75
Crew 3, from 378-to-3398 : 0.25
Crew 5, from 1926-to-838 : 0.25
Crew 2, from 6012-to-1857 : 0.75
Crew 6, from 1731-to-88 : 0.25
Crew 4, from 4889-to-246 : 0.5
Crew 1, from 5523-to-5865 : 0.25
Crew 5, from 838-to-3613 : 0.25
Crew 2, from 1857-to-3754 : 0.25
Crew 4, from 246-to-3019 : 0.25
Crew 6, from 88-to-1402 : 0.25
Crew 3, from 3398-to-4379 : 0.5
C

In [13]:
#pd.DataFrame({"crews": new_controls}).to_excel(
#    "new_controls_"+ ds_sel +".xlsx",
#    index=False
#)

## create an INP file with the new controls (test)

In [14]:
import os


def write_inp_controls(input_inp, output_inp, crews, new_controls):
    """
    Creates a new EPANET INP file using the control rules generated by
    allocate_crews().

    Parameters
    ----------
    input_inp : str
        Path to the original damaged INP file.

    output_inp : str
        Path where the modified INP file will be saved.

    crews : dict
        Crew allocation dictionary returned by allocate_crews().
        (Currently only stored for compatibility and future extensions.)

    new_controls : list of str
        List of EPANET control lines generated by allocate_crews().

    Returns
    -------
    str
        Path to the generated INP file.
    """

    # -------------------------------------------------------------
    # Basic checks
    # -------------------------------------------------------------
    if not os.path.exists(input_inp):
        raise FileNotFoundError(f"Input file not found:\n{input_inp}")

    if not isinstance(new_controls, list):
        raise TypeError("new_controls must be a list of strings.")

    if len(new_controls) == 0:
        raise ValueError("new_controls is empty.")

    # -------------------------------------------------------------
    # Read original INP
    # -------------------------------------------------------------
    with open(input_inp, "r") as f:
        lines = f.readlines()

    # -------------------------------------------------------------
    # Locate [CONTROLS]
    # -------------------------------------------------------------
    controls_start = None

    for i, line in enumerate(lines):
        if line.strip().upper() == "[CONTROLS]":
            controls_start = i
            break

    # -------------------------------------------------------------
    # If [CONTROLS] does not exist, create it
    # -------------------------------------------------------------
    if controls_start is None:

        if lines[-1].strip() != "":
            lines.append("\n")

        lines.append("[CONTROLS]\n")
        controls_start = len(lines) - 1

        controls_end = len(lines)

    else:

        # Find the next section
        controls_end = len(lines)

        for j in range(controls_start + 1, len(lines)):

            txt = lines[j].strip()

            if txt.startswith("[") and txt.endswith("]"):
                controls_end = j
                break

    # -------------------------------------------------------------
    # Build new CONTROLS section
    # -------------------------------------------------------------
    control_block = [
        "[CONTROLS]\n",
        "; ---------------------------------------------------------\n",
        "; Restoration controls generated automatically\n",
        "; ---------------------------------------------------------\n",
    ]

    for control in new_controls:

        control = control.rstrip()

        if control != "":
            control_block.append(control + "\n")

    control_block.append("\n")

    # -------------------------------------------------------------
    # Replace old CONTROLS section
    # -------------------------------------------------------------
    new_lines = (
        lines[:controls_start]
        + control_block
        + lines[controls_end:]
    )

    # -------------------------------------------------------------
    # Save new INP
    # -------------------------------------------------------------
    with open(output_inp, "w") as f:
        f.writelines(new_lines)

    print("=" * 60)
    print("Restoration INP successfully created.")
    print(f"Output file : {output_inp}")
    print(f"Controls    : {len(new_controls)}")
    print("=" * 60)

    return output_inp

In [15]:
# load the wdn as INP file WITH the broken pipes
input_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'

# Export the new INP file with the controls
output_inp="BBM-EPS_"+ds_sel+"_restoration_6c.inp"

In [16]:
controls_inp = write_inp_controls(
    input_inp=input_inp,
    output_inp=output_inp,
    crews=crews,
    new_controls=new_controls
)

print(f"Created: {controls_inp}")

Restoration INP successfully created.
Output file : BBM-EPS_DS5_restoration_6c.inp
Controls    : 432
Created: BBM-EPS_DS5_restoration_6c.inp
